# Lesson 04 - 工具使用設計模式

在本課程中，您將學習使用 Microsoft Agent Framework（Python）為 AI 代理實現的 <strong>工具使用</strong> 設計模式。我們涵蓋：

- 使用 `@tool` 裝飾器和類型參數定義函數工具
- 提供工具架構使模型了解每個工具的功能
- 使用 `approval_mode` 控制工具執行
- 通過 Pydantic 模型和 `response_format` 返回<strong>結構化輸出</strong>

情境是一個 <strong>旅遊預訂代理</strong>，能查詢目的地、檢查可用性及檢索航班資訊。


## 設置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 使用 @tool 裝飾器定義工具

`@tool` 裝飾器將普通的 Python 函數轉換為代理可調用的工具。
要點：

- <strong>文檔字串</strong> 會成為模型看到的工具描述。
- <strong>類型註解</strong>（包括帶描述的 `Annotated`）定義工具的結構。
- `approval_mode` 控制是否必須由使用者批准每次調用後才執行。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## 創建一個具有多個工具的代理

將所有三個工具傳遞給客戶端，讓模型可以調用它需要的任何工具來回答用戶的問題。


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## 使用工具的結構化輸出

透過將 `response_format` 設定為 Pydantic 模型，代理將被強制返回一個型別明確的 JSON 物件，而非自由格式的文字。當下游程式需要以程式化方式使用結果時，這非常有用。


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## 工具批准模式

`@tool` 上的 `approval_mode` 參數控制工具調用是否需要在執行前獲得人工批准：

| 模式 | 行為 |
|---|---|
| `"never_require"` | 工具自動運行 — 無需用戶確認。 |
| `"always_require"` | 每次調用必須經用戶批准後才執行。 |

對於具有副作用的工具（例如預訂航班、收取信用卡費用），請使用 `"always_require"`，以確保人工參與。


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

在本課程中，您學習了如何：

1. 使用帶有類型參數和作為工具結構的 docstring 的 `@tool` 裝飾器來<strong>定義工具</strong>。
2. <strong>組合多個工具</strong>，使代理能夠依序調用它們以回答複雜查詢。
3. 通過傳遞 Pydantic 模型作為 `response_format` 來<strong>返回結構化輸出</strong>。
4. 使用 `approval_mode` <strong>控制工具的審核</strong>，在敏感操作中保持人工監督。

這些模式構成了構建可靠且適合生產環境的代理的基礎，能夠安全地與外部系統互動。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
